# TreeImportanceSelector

Fits a random forest and ranks features by how much they reduce impurity
across all splits.

- **Strength:** sees interactions: features that matter only in combination.
  No scaling needed.
- **Weakness:** costs a forest fit. Favours features with many distinct values,
  and splits importance between correlated features.
- **Requires** `n_features`. More trees (`n_estimators`) give a steadier ranking.

In [1]:
import numpy as np

rng = np.random.default_rng(0)
from featsel.selectors import ANOVAFSelector, TreeImportanceSelector

# The class is the XOR of the signs of x0 and x1: neither alone tells anything.
n = 1000
X = rng.normal(size=(n, 30))
y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(int)

## An interaction that univariate scores cannot see

In [2]:
trees = TreeImportanceSelector(n_features=2, n_estimators=300, random_state=0).fit(X, y)
anova = ANOVAFSelector(n_features=2).fit(X, y)

print('random forest picks:', sorted(trees.get_support(indices=True).tolist()))
print('ANOVA F picks      :', sorted(anova.get_support(indices=True).tolist()))

random forest picks: [0, 1]
ANOVA F picks      : [9, 23]


Each of `x0` and `x1` has the same class mean on both sides, so ANOVA F
picks noise. The forest finds the pair because splitting on one makes the
other useful.

## The importance profile

In [3]:
imp = trees.feature_importances_
print(f'x0 {imp[0]:.3f}  x1 {imp[1]:.3f}  largest noise {imp[2:].max():.3f}')

x0 0.076  x1 0.072  largest noise 0.038
